# Updating data from GenBank

Author: Alexander Maksiaev

Purpose: Update labels from previously gotten data from GISAID + Andersen, using Genbank.

In [1]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "GISAID_Andersen_Combined_Files/"
temp_files = downloads + "Andersen_Temp_Files/"

update_date = "05-12-2025"

os.chdir(originals)

## Collection Dates

In [2]:
# Upload saved data 
os.chdir(temp_files + "saved/")
metadata_genbank = pd.read_csv("metadata_genbank_5-9-2025.csv") # Since 1/1/2024
os.chdir(originals)

display(metadata_genbank)

,Unnamed: 0,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,...,date,File Name,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,name_state,Collection_Date_Specific
0,0,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,...,2025-04-15_06-38-07,SRR33124594.fa,B3.13,"MP:ea1, NS:am1.1, NA:ea1, NP:am8, HA:ea1, PA:e...","ea1:22-003707-003:MP, am1.1:22-010085-001:NS, ...","98.78%, 99.17%, 98.86%, 98.66%, 98.36%, 98.70%...","12, 7, 16, 20, 28, 28, 13, 31",Ran on FASTA - No Coverage Report,USA,2025
1,1,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,...,2025-04-15_06-38-07,SRR33124595.fa,B3.13,"MP:ea1, PB1:am4, NA:ea1, NS:am1.1, NP:am8, HA:...","ea1:22-003707-003:MP, am4:23-001855-001:PB1, e...","98.78%, 99.43%, 98.86%, 99.17%, 98.66%, 98.36%...","12, 13, 16, 7, 20, 28, 28, 31",Ran on FASTA - No Coverage Report,USA,2025
2,2,SRR33124596,WGS,262.12,61927215,PRJNA1102327,SAMN47941262,Viral,21760157,USDA-NVSL,...,2025-04-15_06-38-12,SRR33124596.fa,B3.13,"NP:am8, PB2:am2.2, PB1:am4, HA:ea1, NA:ea1, MP...","am8:23-032005-001:NP, am2.2:22-010445-001:PB2,...","98.80%, 98.55%, 99.43%, 98.42%, 98.86%, 98.88%...","18, 33, 13, 27, 16, 11, 25, 8",Ran on FASTA - No Coverage Report,USA,2025
3,3,SRR33124597,WGS,264.06,73438935,PRJNA1102327,SAMN47941261,Viral,26040226,USDA-NVSL,...,2025-04-15_06-38-07,SRR33124597.fa,B3.13,"PB2:am2.2, HA:ea1, NS:am1.1, PB1:am4, NA:ea1, ...","am2.2:22-010445-001:PB2, ea1:22-003707-003:HA,...","98.64%, 98.30%, 99.17%, 99.38%, 98.86%, 98.88%...","31, 29, 7, 14, 16, 11, 15, 27",Ran on FASTA - No Coverage Report,USA,2025
4,4,SRR33124598,WGS,255.94,76531946,PRJNA1102327,SAMN47941260,Viral,27022104,USDA-NVSL,...,2025-04-15_06-38-07,SRR33124598.fa,B3.13,"PB1:am4, NA:ea1, MP:ea1, HA:ea1, PB2:am2.2, NS...","am4:23-001855-001:PB1, ea1:22-003707-003:NA, e...","99.38%, 98.72%, 98.98%, 98.42%, 98.68%, 99.17%...","14, 18, 10, 27, 30, 7, 27, 18",Ran on FASTA - No Coverage Report,USA,2025
5,5,SRR33124599,WGS,145.79,94820034,PRJNA1102327,SAMN47941301,Viral,34313048,USDA-NVSL,...,2025-04-15_06-38-12,SRR33124599.fa,B3.13,"HA:ea1, NP:am8, PB2:am2.2, NS:am1.1, NA:ea1, P...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.53%, 98.93%, 98.60%, 99.05%, 98.94%, 99.02%...","25, 16, 32, 8, 15, 21, 11, 12",Ran on FASTA - No Coverage Report,USA,2025-02-22
6,6,SRR33124600,WGS,147.14,130133538,PRJNA1102327,SAMN47941300,Viral,47111937,USDA-NVSL,...,2025-04-15_06-38-07,SRR33124600.fa,B3.13,"NA:ea1, HA:ea1, PB2:am2.2, PB1:am4, NS:am1.1, ...","ea1:22-003707-003:NA, ea1:22-003707-003:HA, am...","98.94%, 98.53%, 98.60%, 99.47%, 99.05%, 99.02%...","15, 25, 32, 12, 8, 21, 11, 16",Ran on FASTA - No Coverage Report,USA,2025-02-22
7,7,SRR33124602,WGS,248.60,93818995,PRJNA1102327,SAMN47941298,Viral,32805554,USDA-NVSL,...,2025-04-15_06-38-12,SRR33124602.fa,B3.13,"PA:ea1, PB1:am4, HA:ea1, MP:ea1, NA:ea1, NS:am...","ea1:22-003707-003:PA, am4:23-001855-001:PB1, e...","99.02%, 99.52%, 98.59%, 98.78%, 98.94%, 99.05%...","21, 11, 24, 12, 15, 8, 32, 16",Ran on FASTA - No Coverage Report,USA,2025-02-19
8,8,SRR33124603,WGS,232.01,82101812,PRJNA1102327,SAMN47941297,Viral,27946678,USDA-NVSL,...,2025-04-15_06-38-07,SRR33124603.fa,B3.13,"PB2:am2.2, PB1:am4, MP:ea1, NS:am1.1, PA:ea1, ...","am2.2:22-010445-001:PB2, am4:23-001855-001:PB1...","98.55%, 99.47%, 98.88%, 99.05%, 98.98%, 98.59%...","33, 12, 11, 8, 22, 24, 16, 15",Ran on FASTA - No Coverage Report,USA,2025
9,9,SRR33124604,WGS,147.68,102946466,PRJNA1102327,SAMN47941296,Viral,37576956,USDA-NVSL,...,2025-04-15_06-38-15,SRR33124604.fa,B3.13,"PB2:am2.2, PA:ea1, NP:am8, NA:ea1, NS:am1.1, H...","am2.2:22-010445-001:PB2, ea1:22-003707-003:PA,...","98.51%, 99.02%, 98.93%, 98.79%, 98.69%, 98.30%...","34, 21, 16, 17, 11, 29, 16, 10",Ran on FASTA - No Coverage Report,USA,2025


In [3]:
# no_updates = pd.DataFrame()
# no_updates_isolate = []

# Get only labels that have no states or collection dates, and update them

def update(file_name, update_date):
    updates = {}
    with open(file_name) as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if line[0] == ">": # It's a header
                header = line 
                collection_date = header.split("|")[-3]
                state = header.split("/")[2]
                state = state.replace(": ", "-")
                geo_location_collection_date = state + "|" + collection_date
                isolate = header.split("/")[3]
                sequence = lines[i + 1] # Sequence always comes in one line after header
                # if "-" not in collection_date: # If there are no dashes, i.e. if it's just the year
                    # Find the correct collection date, if it exists
                row = metadata_genbank[metadata_genbank["isolate"] == isolate]
                
                try: # Isolate may not be in this dataset
                    id = row["BioSample"].values[0]
                    geo_location_collection_date = search_collection_date(id, row) # Update unknown dates, if possible
                except:
                    print("No BioSample nor date nor state found for isolate", isolate)
                        # no_updates_isolate.append(isolate)

                # if state == "USA": # If we don't have a state
                #     row = metadata_genbank[metadata_genbank["isolate"] == isolate]

                #     try: # Isolate may not be in this dataset
                #         genbank_name = row["genbank_name"].values[0]
                #         state = genbank_name.split("/")[2]
                #     except:
                #         print("No state found for isolate", isolate)
                #         # no_updates_isolate.append(isolate)

                updates[header] = [geo_location_collection_date, sequence]

        f.close()

    updates_df = pd.DataFrame.from_dict(updates, orient="index", columns=["geo_location_collection_date", "sequence"])
    updates_df["header"] = updates_df.index
    updates_df = updates_df.reset_index()

    updated_file_name = ".".join(file_name.split(".")[:-1]) + "_" + update_date + "_update." + file_name.split(".")[-1]

    with open(updated_file_name, "w") as g:

        for i, row in updates_df.iterrows():
            header = row["header"]
            # print(header)
            # print(header.split("|")[-3])
            
            header = header.replace(str(header.split("|")[-3]), str(row["geo_location_collection_date"])) # Only the first instance is replaced

            g.write(header)
            g.write(row["sequence"])

        g.close()

    # no_updates["isolate"] = no_updates_isolate
    # no_updates.to_csv("not_updated.csv")


In [4]:
# Create files with updates

os.chdir(originals)

# for dirpath, dirs, files in os.walk(originals + "04-14-2025--05-09-2025_B3_13/"): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         update(file_name, update_date)
#     break 



In [5]:
search_collection_date_term("PP752829.1", metadata_genbank)

PP752829.1
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_68223f127728d776d5053dfc&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024


'Texas|20-Mar-2024'